# Étape 4 : diversification et couverture par des actifs extérieurs au thème

Ce carnet applique les douze stratégies fixées dans la phase 0 de l'étape 4, section VII de `research/plan_projet.md`, écrite avant tout calcul, aux vingt séries de l'étape 2 et au témoin `T1` rééquilibré. Les fonctions viennent de `src/couverture.py` et de `src/risque.py`, testées dans `tests/test_couverture.py` et `tests/test_risque.py`. Les instruments sont collectés par `python -m src.collecter_couverture`.

Les treize lignes par série sont : la série non couverte ; 80 % et 60 % de la série mêlés à des liquidités, à des obligations du Trésor intermédiaires (`VFITX`), longues (`VUSTX`) ou au témoin `T1` ; la vente de contrats à terme sur la moitié ou la totalité du bêta ; l'achat mensuel de puts à 90 % ou 95 % du cours sur la totalité du bêta.

In [1]:
import sys, gzip, json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

RACINE = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists())
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

from src import risque
from src import couverture as cv
from src.collecter_couverture import verifier
from src.empreintes import empreinte

SORTIE = RACINE / "data" / "processed"
BRUT = RACINE / "data" / "raw"
manifeste_instruments = verifier(RACINE)

valeurs = pd.read_csv(SORTIE / "valeurs_portefeuilles.csv", index_col=0)
temoin = pd.read_csv(SORTIE / "temoin_valeurs.csv", index_col=0)
comparaisons = pd.read_csv(SORTIE / "valeurs_comparaisons.csv", index_col=0)
dates = valeurs.index
series = {**{s: valeurs[s] for s in valeurs.columns}, "T1_reeq": temoin["T1_reeq"]}

taux = pd.read_csv(BRUT / "taux_sans_risque.csv", index_col=0)["taux_annuel_pct"].reindex(dates).ffill()
sans_risque = (1 + taux / 100) ** (1 / 252) - 1
taux_continu = np.log1p(taux / 100)

r_spy = comparaisons["SPY"].pct_change()
excedent_marche = r_spy - sans_risque

def lire_indice(chemin):
    df = pd.read_csv(chemin)
    df.index = pd.Index(df.date.str[:10], name="date")
    return df

gspc = lire_indice(BRUT / "benchmarks" / "GSPC.csv")["Close"].reindex(dates)
spy_brut = lire_indice(BRUT / "benchmarks" / "SPY.csv")
dividendes = spy_brut["Dividends"].copy()
dividendes.index = pd.to_datetime(dividendes.index)
rendement_dividende = (dividendes.rolling("365D").sum() / spy_brut["Close"].set_axis(dividendes.index))
rendement_dividende.index = rendement_dividende.index.strftime("%Y-%m-%d")
q = np.log1p(rendement_dividende.reindex(dates).ffill())

instruments = {s: pd.read_csv(BRUT / "couverture" / f, index_col=0)
               for s, f in [("VFITX", "VFITX.csv"), ("VUSTX", "VUSTX.csv"), ("VIX", "VIX.csv")]}
r_inter = instruments["VFITX"]["Adj Close"].pct_change().reindex(dates)
r_long = instruments["VUSTX"]["Adj Close"].pct_change().reindex(dates)
vix = instruments["VIX"]["Close"].reindex(dates) / 100
r_t1 = temoin["T1_reeq"].pct_change()

annuel = cv.premieres_seances_annee(dates)
fins = cv.fins_de_mois(dates)
trimestres = {d for d in fins if d[5:7] in {"03", "06", "09", "12"}}
annees = (len(dates) - 1) / 252

episodes = pd.read_csv(SORTIE / "episodes_tension.csv")
tension = pd.Series(False, index=dates)
for e in episodes.itertuples():
    tension.loc[e.debut:e.creux] = True

print(len(series), "series,", len(dates), "seances, du", dates[0], "au", dates[-1])
print("valeurs absentes : GSPC %d, VIX %d, VFITX %d, VUSTX %d, q %d" % (
    gspc.isna().sum(), vix.isna().sum(), r_inter.iloc[1:].isna().sum(), r_long.iloc[1:].isna().sum(), q.isna().sum()))
print("rendement du dividende de SPY : median %.2f %%, min %.2f %%, max %.2f %%"
      % (100 * q.median(), 100 * q.min(), 100 * q.max()))

21 series, 6709 seances, du 2000-01-03 au 2026-09-04
valeurs absentes : GSPC 0, VIX 0, VFITX 0, VUSTX 0, q 0
rendement du dividende de SPY : median 1.74 %, min 0.84 %, max 3.92 %


## 1. Contrôle des instruments

Avant de construire quoi que ce soit, je regarde les variations quotidiennes extrêmes des deux fonds obligataires et l'étendue du `VIX`. Un fonds du Trésor qui bouge de plus de 3 % en une séance est rare ; chaque cas doit correspondre à un événement de taux connu.

In [2]:
for nom, r in [("VFITX", r_inter), ("VUSTX", r_long)]:
    extremes = r[r.abs() > 0.03]
    print(f"{nom} : volatilite annualisee {risque.volatilite(r.dropna()):.2%}, "
          f"rendement annualise {risque.annualiser_rendement((1 + r.fillna(0)).cumprod()):.2%}, "
          f"{len(extremes)} seances au-dela de 3 %")
    print(extremes.round(4).to_string() if len(extremes) else "  aucune")
print()
print("VIX : min %.1f, median %.1f, max %.1f, le %s" % (100 * vix.min(), 100 * vix.median(), 100 * vix.max(), vix.idxmax()))

VFITX : volatilite annualisee 4.89%, rendement annualise 3.97%, 0 seances au-dela de 3 %
  aucune
VUSTX : volatilite annualisee 12.32%, rendement annualise 4.40%, 19 seances au-dela de 3 %
date
2008-11-20    0.0319
2009-03-18    0.0447
2011-08-11   -0.0394
2011-09-22    0.0324
2011-10-31    0.0353
2013-07-05   -0.0321
2016-11-09   -0.0391
2020-03-06    0.0498
2020-03-10   -0.0487
2020-03-16    0.0510
2020-03-17   -0.0587
2020-03-18   -0.0434
2020-03-20    0.0652
2020-03-23    0.0419
2021-02-26    0.0323
2022-03-02   -0.0319
2022-09-28    0.0317
2022-11-10    0.0372
2023-03-10    0.0314

VIX : min 9.1, median 17.8, max 82.7, le 2020-03-16


## 2. Construction des treize lignes par série

Le bêta de chaque série sur `SPY` est estimé sur les 252 séances précédentes, arrêté la veille. Pendant les 63 premières séances de 2000, faute d'historique, il vaut un. **Complément du 28 septembre 2026 à la phase 0, écrit avant tout résultat** : la phase 0 ne disait pas quel bêta retenir avant que 252 séances soient disponibles, alors que les séries commencent le 3 janvier 2000. J'estime le bêta dès 63 séances, sur toutes celles qui existent, et je retiens un avant ce seuil.

Pour les puts, la volatilité vaut le `VIX` du jour plus trois points, la fourchette 5 % de la prime.

In [3]:
MELANGES = [("liquidites", sans_risque), ("treso_inter", r_inter), ("treso_long", r_long), ("hors_theme", r_t1)]
PENTE, FOURCHETTE = 0.03, cv.FOURCHETTE_PUT


def puts(r, beta, niveau_k, pente=PENTE, fourchette=FOURCHETTE):
    return cv.protection_put(r, gspc, vix + pente, taux_continu, q, beta, fins, niveau_k, fourchette)


def construire(nom, niveau):
    r = niveau.pct_change()
    beta = cv.beta_glissant(r, r_spy)
    lignes = {"non_couvert": (100 * niveau / niveau.iloc[0], 0.0, 0.0)}
    for actif, r_autre in MELANGES:
        if actif == "hors_theme" and nom.startswith("T1"):
            continue
        for part in (0.8, 0.6):
            v, frais = cv.melange(r, r_autre, part, annuel)
            lignes[f"{actif}_{round(100 * part)}"] = (v, frais, 0.0)
    for part in (0.5, 1.0):
        v, frais = cv.couverture_futur(r, excedent_marche, beta, part, fins, trimestres)
        lignes[f"futur_{round(100 * part)}"] = (v, frais, 0.0)
    for niveau_k in (0.90, 0.95):
        v, primes, encaisse = puts(r, beta, niveau_k)
        lignes[f"put_{round(100 * niveau_k)}"] = (v, primes, encaisse)
    return lignes, beta


resultats, betas = {}, {}
for nom, niveau in series.items():
    resultats[nom], betas[nom] = construire(nom, niveau)

STRATEGIES = list(resultats["P1_reeq"])
table = pd.DataFrame({f"{s}|{k}": v for s, lignes in resultats.items() for k, (v, _, _) in lignes.items()})
print(table.shape, "colonnes serie|strategie")
print(pd.DataFrame(betas).describe().loc[["min", "50%", "max"], ["P1_reeq", "P1_cons", "P5_reeq", "P6_reeq", "T1_reeq"]].round(3).to_string())

(6709, 271) colonnes serie|strategie
     P1_reeq  P1_cons  P5_reeq  P6_reeq  T1_reeq
min    0.591    0.591    0.824    0.021    0.490
50%    1.078    1.094    1.337    0.606    0.981
max    1.458    2.037    2.450    1.027    1.192


## 3. Contrôles de la construction

Quatre identités doivent tenir. Un mélange à 100 % de la série redonne la série. Chaque valeur reste strictement positive. La prime d'un put à un mois à 90 % du cours doit rester dans l'ordre de grandeur connu, quelques dixièmes de pour cent du nominal par mois au calme, davantage en crise. Et la couverture par contrats à terme d'une série égale au marché ne doit laisser que le taux sans risque, cas vérifié dans les tests.

In [4]:
ecarts = []
for nom, niveau in series.items():
    v, frais = cv.melange(niveau.pct_change(), sans_risque, 1.0, annuel)
    ecarts.append(float((v / (100 * niveau / niveau.iloc[0]) - 1).abs().max()))
print("melange a 100 %% : ecart relatif maximal %.2e sur %d series" % (max(ecarts), len(ecarts)))
print("valeurs non positives :", int((table <= 0).sum().sum()), "| valeurs absentes :", int(table.isna().sum().sum()))

jours_achat = [d for d in dates if d in fins] + [dates[0]]
achat = pd.DataFrame({"S": gspc, "sigma": vix + PENTE, "r": taux_continu, "q": q}).loc[sorted(jours_achat)]
for niveau_k in (0.90, 0.95):
    prime = cv.put_black_scholes(achat.S, niveau_k * achat.S, 21 / 252, achat.r, achat.q, achat.sigma) / achat.S
    print(f"put a {niveau_k:.0%} : prime mensuelle en % du nominal, mediane {100 * np.median(prime):.3f}, "
          f"max {100 * prime.max():.2f}, cout annuel median {1200 * np.median(prime):.2f} pts")
    fin_cours = gspc.loc[sorted(jours_achat)]
    dans_monnaie = (fin_cours.shift(-1) < niveau_k * fin_cours).sum()
    print(f"   echeances dans la monnaie : {dans_monnaie} sur {len(fin_cours) - 1}")

melange a 100 % : ecart relatif maximal 2.56e-14 sur 21 series
valeurs non positives : 0 | valeurs absentes : 0


put a 90% : prime mensuelle en % du nominal, mediane 0.085, max 3.04, cout annuel median 1.02 pts
   echeances dans la monnaie : 4 sur 321
put a 95% : prime mensuelle en % du nominal, mediane 0.614, max 4.90, cout annuel median 7.37 pts
   echeances dans la monnaie : 37 sur 321


### Contrôle du niveau des prix d'option contre un indice publié

*Contrôle ajouté le 28 septembre 2026, après la construction et avant la rédaction des résultats.* La phase 0 annonçait que les prix de modèle sous-estimaient probablement le coût réel des puts. Je peux le vérifier sur un point. L'indice CBOE S&P 500 PutWrite, `^PUT`, vend chaque mois des puts à la monnaie sur le S&P 500 garantis par des bons du Trésor, aux prix réellement cotés. Si je réplique cette vente avec mon modèle, l'écart de rendement annuel entre la réplique et l'indice dit dans quel sens mon modèle se trompe sur le niveau des primes. Le renouvellement de l'indice se fait le troisième vendredi, le mien en fin de mois : les rendements quotidiens ne coïncident donc pas, seul le niveau sur vingt-six ans est comparé.

In [5]:
indice_put = pd.read_csv(BRUT / "couverture" / "PUT.csv", index_col=0)["Close"].reindex(dates).ffill()
cible = risque.annualiser_rendement(indice_put)
print("indice PutWrite : %.2f %% par an ; bons du Tresor : %.2f %% par an"
      % (100 * cible, 100 * risque.annualiser_rendement((1 + sans_risque.fillna(0)).cumprod())))

def replique(pente):
    return cv.vente_puts_garantie(gspc, vix + pente, taux_continu, q, sans_risque, fins, 1.0)

for pente in [-0.03, 0.0, PENTE]:
    s = replique(pente)
    print(f"  replique, VIX {100 * pente:+.0f} points : {risque.annualiser_rendement(s):.2%} par an, "
          f"correlation quotidienne avec l'indice {s.pct_change().corr(indice_put.pct_change()):.3f}")

bas, haut = -0.10, 0.0
while haut - bas > 1e-4:
    milieu = (bas + haut) / 2
    if risque.annualiser_rendement(replique(milieu)) > cible:
        haut = milieu
    else:
        bas = milieu
PENTE_CALIBREE = round((bas + haut) / 2, 4)
print(f"volatilite a la monnaie qui egalise la replique et l'indice : VIX {100 * PENTE_CALIBREE:+.2f} points")

indice PutWrite : 6.93 % par an ; bons du Tresor : 1.90 % par an


  replique, VIX -3 points : 8.15% par an, correlation quotidienne avec l'indice 0.832


  replique, VIX +0 points : 12.70% par an, correlation quotidienne avec l'indice 0.832


  replique, VIX +3 points : 17.41% par an, correlation quotidienne avec l'indice 0.833


volatilite a la monnaie qui egalise la replique et l'indice : VIX -3.82 points


## 4. Les mesures, sur toutes les séries et toutes les stratégies

Les neuf mesures de l'étape 3, aux mêmes conventions, et trois ajouts de la phase 0 : le rendement abandonné face à la série non couverte, le coût explicite annuel, et le bêta à `SPY` réalisé. Le coût explicite est le total des frais, ou des primes payées moins les montants encaissés à l'échéance, rapporté à la valeur moyenne et au nombre d'années.

In [6]:
def mois(niveau):
    return niveau.groupby(pd.Series(niveau.index).str[:7].values).last().pct_change().dropna()


def mesures(niveau):
    r = niveau.pct_change().dropna()
    excedents = (r - sans_risque.reindex(r.index)).dropna()
    m = r_spy.reindex(r.index)
    ok = m.notna()
    repli = risque.drawdown(niveau)
    return {"annualise": risque.annualiser_rendement(niveau),
            "vol_quotidienne": risque.volatilite(r),
            "vol_mensuelle": risque.volatilite(mois(niveau), risque.MOIS),
            "semi_volatilite": risque.volatilite_baisse(excedents),
            "repli_max": repli["repli_max"], "chute": repli["chute"], "recuperation": repli["recuperation"],
            "asymetrie": r.skew(), "aplatissement": r.kurt(),
            "var95": risque.var_historique(r, 0.95), "es95": risque.perte_au_dela(r, 0.95),
            "var99": risque.var_historique(r, 0.99), "es99": risque.perte_au_dela(r, 0.99),
            "sharpe": risque.sharpe(r, sans_risque), "sortino": risque.sortino(r, sans_risque),
            "beta_spy": float(np.polyfit(m[ok], r[ok], 1)[0])}


lignes = []
for nom, strategies in resultats.items():
    for strat, (v, paye, encaisse) in strategies.items():
        lignes.append({"serie": nom, "strategie": strat, **mesures(v),
                       "cout_explicite": (paye - encaisse) / v.mean() / annees})
mesures_df = pd.DataFrame(lignes)
base = mesures_df[mesures_df.strategie == "non_couvert"].set_index("serie")
mesures_df["rendement_abandonne"] = mesures_df.serie.map(base.annualise) - mesures_df.annualise
for col in ["vol_quotidienne", "repli_max", "es99"]:
    mesures_df[f"reduction_{col}"] = 1 - mesures_df[col] / mesures_df.serie.map(base[col])
mesures_df.to_csv(SORTIE / "couverture_mesures.csv", index=False, encoding="utf-8")

colonnes = ["annualise", "vol_quotidienne", "vol_mensuelle", "repli_max", "es99", "sharpe", "beta_spy",
            "rendement_abandonne", "cout_explicite"]
for nom in ["P1_reeq", "P1_cons", "T1_reeq"]:
    print("===", nom)
    print(mesures_df[mesures_df.serie == nom].set_index("strategie")[colonnes].round(4).to_string())
    print()

=== P1_reeq
                annualise  vol_quotidienne  vol_mensuelle  repli_max    es99  sharpe  beta_spy  rendement_abandonne  cout_explicite
strategie                                                                                                                          
non_couvert        0.1870           0.2173         0.1910    -0.5184  0.0510  0.8115    1.0496               0.0000          0.0000
liquidites_80      0.1567           0.1719         0.1529    -0.4268  0.0394  0.8239    0.8247               0.0303          0.0001
liquidites_60      0.1248           0.1282         0.1151    -0.3309  0.0288  0.8348    0.6101               0.0622          0.0001
treso_inter_80     0.1617           0.1688         0.1512    -0.4137  0.0383  0.8616    0.8058               0.0253          0.0001
treso_inter_60     0.1343           0.1237         0.1132    -0.3035  0.0273  0.9286    0.5760               0.0527          0.0001
treso_long_80      0.1638           0.1655         0.1516    -0.

### Lecture sur les vingt séries

Pour chaque stratégie, le nombre de séries sur vingt où elle réduit la volatilité, le repli maximal et la perte moyenne au-delà de la VaR à 99 %, et les médianes des réductions et du rendement abandonné.

In [7]:
vingt = mesures_df[mesures_df.serie.str.startswith("P") & (mesures_df.strategie != "non_couvert")]
synthese = vingt.groupby("strategie", sort=False).agg(
    baisse_vol=("reduction_vol_quotidienne", lambda x: int((x > 0).sum())),
    baisse_repli=("reduction_repli_max", lambda x: int((x > 0).sum())),
    baisse_es99=("reduction_es99", lambda x: int((x > 0).sum())),
    med_red_vol=("reduction_vol_quotidienne", "median"),
    med_red_repli=("reduction_repli_max", "median"),
    med_red_es99=("reduction_es99", "median"),
    med_abandon=("rendement_abandonne", "median"),
    med_cout=("cout_explicite", "median"))
synthese.to_csv(SORTIE / "couverture_synthese.csv", encoding="utf-8")
print(synthese.round(4).to_string())

                baisse_vol  baisse_repli  baisse_es99  med_red_vol  med_red_repli  med_red_es99  med_abandon  med_cout
strategie                                                                                                             
liquidites_80           20            20           20       0.2088         0.1671        0.2215       0.0239    0.0001
liquidites_60           20            20           20       0.4084         0.3539        0.4258       0.0502    0.0001
treso_inter_80          20            20           20       0.2197         0.1957        0.2349       0.0186    0.0001
treso_inter_60          20            20           20       0.4235         0.4044        0.4431       0.0403    0.0001
treso_long_80           20            20           20       0.2294         0.2067        0.2434       0.0169    0.0001
treso_long_60           20            20           20       0.4241         0.4269        0.4379       0.0367    0.0001
hors_theme_80           20            17        

## 5. Les diversifiants face aux liquidités à part égale

C'est la comparaison décisive de la phase 0 : à proportion égale, un actif ne diversifie que s'il fait mieux que des liquidités. L'écart est calculé sur la volatilité, la perte au-delà de la VaR à 99 % et le repli maximal ; un écart négatif signifie que l'actif réduit davantage le risque que les liquidités.

In [8]:
lignes = []
for nom in series:
    bloc = mesures_df[mesures_df.serie == nom].set_index("strategie")
    for actif in ["treso_inter", "treso_long", "hors_theme"]:
        for part in (80, 60):
            if f"{actif}_{part}" not in bloc.index:
                continue
            a, l = bloc.loc[f"{actif}_{part}"], bloc.loc[f"liquidites_{part}"]
            lignes.append({"serie": nom, "actif": actif, "part_serie": part,
                           "ecart_vol": a.vol_quotidienne - l.vol_quotidienne,
                           "ecart_es99": a.es99 - l.es99, "ecart_repli": a.repli_max - l.repli_max,
                           "ecart_rendement": a.annualise - l.annualise})
face_liquidites = pd.DataFrame(lignes)
face_liquidites.to_csv(SORTIE / "couverture_face_liquidites.csv", index=False, encoding="utf-8")
p = face_liquidites[face_liquidites.serie.str.startswith("P")]
print(p.groupby(["actif", "part_serie"]).agg(
    vol_plus_basse=("ecart_vol", lambda x: int((x < 0).sum())),
    es99_plus_basse=("ecart_es99", lambda x: int((x < 0).sum())),
    repli_moins_profond=("ecart_repli", lambda x: int((x > 0).sum())),
    med_ecart_vol=("ecart_vol", "median"), med_ecart_es99=("ecart_es99", "median"),
    med_ecart_rendement=("ecart_rendement", "median")).round(4).to_string())
print()
print(face_liquidites[face_liquidites.serie.isin(["P1_reeq", "P1_cons", "T1_reeq"])].round(4).to_string(index=False))

                        vol_plus_basse  es99_plus_basse  repli_moins_profond  med_ecart_vol  med_ecart_es99  med_ecart_rendement
actif       part_serie                                                                                                          
hors_theme  60                       0                0                    1         0.0642          0.0184               0.0533
            80                       0                0                    4         0.0320          0.0092               0.0261
treso_inter 60                      20               20                   20        -0.0041         -0.0011               0.0095
            80                      20               20                   20        -0.0028         -0.0008               0.0050
treso_long  60                      15               20                   20        -0.0050         -0.0013               0.0126
            80                      20               20                   20        -0.0057      

## 6. Les corrélations avec les actifs ajoutés, au calme, en tension, avant et depuis 2022

Les deux sous-périodes sont celles de la phase 0, fixées sur la foi du rapport du FMI d'avril 2026 et non des données de cette étude.

In [9]:
AVANT, DEPUIS = slice(None, "2021-12-31"), slice("2022-01-01", None)
actifs = {"treso_inter": r_inter, "treso_long": r_long, "hors_theme": r_t1, "marche": r_spy}
lignes = []
for nom, niveau in series.items():
    r = niveau.pct_change()
    for actif, ra in actifs.items():
        if actif == "hors_theme" and nom.startswith("T1"):
            continue
        paire = pd.concat([r, ra], axis=1).dropna()
        paire.columns = ["s", "a"]
        t = tension.reindex(paire.index)
        lignes.append({"serie": nom, "actif": actif,
                       "complete": paire.s.corr(paire.a),
                       "calme": paire[~t].s.corr(paire[~t].a), "tension": paire[t].s.corr(paire[t].a),
                       "avant_2022": paire.loc[AVANT].s.corr(paire.loc[AVANT].a),
                       "depuis_2022": paire.loc[DEPUIS].s.corr(paire.loc[DEPUIS].a)})
correlations = pd.DataFrame(lignes)
correlations.to_csv(SORTIE / "couverture_correlations.csv", index=False, encoding="utf-8")
print(correlations[correlations.serie.isin(["P1_reeq", "P1_cons", "T1_reeq"])].round(3).to_string(index=False))
print()
print("medianes sur les vingt series :")
print(correlations[correlations.serie.str.startswith("P")].groupby("actif")
      [["complete", "calme", "tension", "avant_2022", "depuis_2022"]].median().round(3).to_string())

  serie       actif  complete  calme  tension  avant_2022  depuis_2022
P1_reeq treso_inter    -0.246 -0.197   -0.324      -0.334        0.093
P1_reeq  treso_long    -0.256 -0.237   -0.309      -0.351        0.109
P1_reeq  hors_theme     0.902  0.887    0.922       0.930        0.754
P1_reeq      marche     0.923  0.914    0.936       0.930        0.888
P1_cons treso_inter    -0.225 -0.176   -0.306      -0.319        0.032
P1_cons  treso_long    -0.235 -0.208   -0.302      -0.338        0.048
P1_cons  hors_theme     0.841  0.798    0.902       0.922        0.610
P1_cons      marche     0.893  0.875    0.922       0.925        0.845
T1_reeq treso_inter    -0.254 -0.213   -0.318      -0.341        0.133
T1_reeq  treso_long    -0.270 -0.262   -0.302      -0.365        0.152
T1_reeq      marche     0.937  0.935    0.941       0.947        0.876

medianes sur les vingt series :
             complete  calme  tension  avant_2022  depuis_2022
actif                                               

## 7. Les deux sous-périodes

Chaque stratégie est relue sur chaque sous-période, la valeur étant rebasée au début de la sous-période. Les stratégies ont été construites sur toute la période, et la relecture ne change que la fenêtre de mesure.

In [10]:
def mesures_courtes(niveau):
    r = niveau.pct_change().dropna()
    return {"annualise": risque.annualiser_rendement(niveau), "vol_quotidienne": risque.volatilite(r),
            "repli_max": risque.drawdown(niveau)["repli_max"], "es99": risque.perte_au_dela(r, 0.99),
            "sharpe": risque.sharpe(r, sans_risque)}

lignes = []
for periode, fenetre in [("avant_2022", AVANT), ("depuis_2022", DEPUIS), ("depuis_2004-08-19", slice("2004-08-19", None))]:
    for nom, strategies in resultats.items():
        if periode.startswith("depuis_2004") and not nom.startswith("P4"):
            continue
        for strat, (v, _, _) in strategies.items():
            lignes.append({"periode": periode, "serie": nom, "strategie": strat, **mesures_courtes(v.loc[fenetre])})
sous_periodes = pd.DataFrame(lignes)
ref = sous_periodes[sous_periodes.strategie == "non_couvert"].set_index(["periode", "serie"])
cle = pd.MultiIndex.from_frame(sous_periodes[["periode", "serie"]])
for col in ["vol_quotidienne", "es99", "repli_max"]:
    sous_periodes[f"reduction_{col}"] = 1 - sous_periodes[col].values / ref[col].reindex(cle).values
sous_periodes["rendement_abandonne"] = ref["annualise"].reindex(cle).values - sous_periodes.annualise.values
sous_periodes.to_csv(SORTIE / "couverture_sous_periodes.csv", index=False, encoding="utf-8")
vue = sous_periodes[(sous_periodes.serie == "P1_reeq")].pivot(index="strategie", columns="periode",
      values=["reduction_vol_quotidienne", "reduction_es99", "rendement_abandonne"])
print(vue.round(4).to_string())

               reduction_vol_quotidienne             reduction_es99             rendement_abandonne            
periode                       avant_2022 depuis_2022     avant_2022 depuis_2022          avant_2022 depuis_2022
strategie                                                                                                      
futur_100                         0.6328      0.5442         0.6620      0.4394              0.0776      0.1075
futur_50                          0.4121      0.3620         0.4438      0.3177              0.0359      0.0508
hors_theme_60                     0.0598      0.1437         0.0179      0.1239             -0.0023      0.0659
hors_theme_80                     0.0324      0.0752         0.0116      0.0783             -0.0013      0.0329
liquidites_60                     0.4149      0.3870         0.4443      0.3860              0.0571      0.0883
liquidites_80                     0.2124      0.1926         0.2332      0.1927              0.0276     

## 8. Les six épisodes de tension, un par un

Rendement de chaque stratégie du sommet au creux de `SPY`. Aucune moyenne n'est calculée sur les épisodes, qui ne sont pas comparables entre eux.

In [11]:
lignes = []
for e in episodes.itertuples():
    for nom, strategies in resultats.items():
        for strat, (v, _, _) in strategies.items():
            lignes.append({"episode": e.debut[:4] + "-" + e.creux[:4], "debut": e.debut, "creux": e.creux,
                           "serie": nom, "strategie": strat, "rendement": v.loc[e.creux] / v.loc[e.debut] - 1})
par_episode = pd.DataFrame(lignes)
par_episode.to_csv(SORTIE / "couverture_episodes.csv", index=False, encoding="utf-8")
print(par_episode[par_episode.serie == "P1_reeq"].pivot(index="strategie", columns="debut", values="rendement")
      .round(3).to_string())
print()
print(par_episode[par_episode.serie == "P1_cons"].pivot(index="strategie", columns="debut", values="rendement")
      .round(3).to_string())

debut           2000-03-24  2007-10-09  2018-09-20  2020-02-19  2022-01-03  2025-02-19
strategie                                                                             
futur_100            0.251       0.140      -0.000      -0.015       0.066       0.021
futur_50            -0.108      -0.236      -0.094      -0.201      -0.070      -0.101
hors_theme_60       -0.178      -0.506      -0.188      -0.372      -0.188      -0.189
hors_theme_80       -0.282      -0.503      -0.187      -0.370      -0.192      -0.201
liquidites_60       -0.200      -0.315      -0.112      -0.224      -0.114      -0.129
liquidites_80       -0.290      -0.411      -0.149      -0.296      -0.155      -0.172
non_couvert         -0.376      -0.500      -0.185      -0.368      -0.197      -0.214
put_90              -0.423      -0.507      -0.142      -0.244      -0.228      -0.159
put_95              -0.409      -0.454      -0.096      -0.177      -0.198      -0.130
treso_inter_60      -0.109      -0.271     

## 9. Les trois hypothèses écrites avant calcul

Chacune est confrontée à son critère de contradiction, tel qu'il est écrit dans la tâche 8 de la phase 0.

In [12]:
sp = sous_periodes.set_index(["periode", "serie", "strategie"])
print("Hypothese 1 : obligations contre liquidites a part egale, sur les vingt series")
for actif in ["treso_inter", "treso_long"]:
    for part in (80, 60):
        for periode in ["avant_2022", "depuis_2022"]:
            vol = es = 0
            for nom in valeurs.columns:
                a, l = sp.loc[(periode, nom, f"{actif}_{part}")], sp.loc[(periode, nom, f"liquidites_{part}")]
                vol += a.vol_quotidienne < l.vol_quotidienne
                es += a.es99 < l.es99
            print(f"  {actif}_{part} {periode:>12} : volatilite plus basse que les liquidites {vol}/20, "
                  f"perte au-dela plus basse {es}/20")

print()
print("Hypothese 2 : actions hors theme a 60 % contre liquidites a 60 %, baisse de volatilite")
for nom in ["P1_reeq", "P1_cons"]:
    b = mesures_df[mesures_df.serie == nom].set_index("strategie")
    print(f"  {nom} : hors theme {b.loc['hors_theme_60', 'reduction_vol_quotidienne']:.1%}, "
          f"liquidites {b.loc['liquidites_60', 'reduction_vol_quotidienne']:.1%}")
compte = sum(mesures_df.set_index(["serie", "strategie"]).loc[(n, "hors_theme_60"), "reduction_vol_quotidienne"]
             > mesures_df.set_index(["serie", "strategie"]).loc[(n, "liquidites_60"), "reduction_vol_quotidienne"]
             for n in valeurs.columns)
print(f"  series ou le hors theme reduit davantage la volatilite que les liquidites : {compte}/20")

print()
print("Hypothese 3 : P1 couvert par contrats a terme en 2000-2002, perte contre la moitie de la perte non couverte")
pe = par_episode.set_index(["debut", "serie", "strategie"])
for nom in ["P1_reeq", "P1_cons"]:
    nu = pe.loc[("2000-03-24", nom, "non_couvert"), "rendement"]
    for s in ["futur_50", "futur_100", "put_90", "put_95"]:
        c = pe.loc[("2000-03-24", nom, s), "rendement"]
        print(f"  {nom} {s:>9} : {c:+.1%} contre {nu:+.1%} non couvert ; contradiction si perte < {nu / 2:+.1%} : "
              f"{'oui' if c > nu / 2 else 'non'}")

Hypothese 1 : obligations contre liquidites a part egale, sur les vingt series
  treso_inter_80   avant_2022 : volatilite plus basse que les liquidites 20/20, perte au-dela plus basse 20/20
  treso_inter_80  depuis_2022 : volatilite plus basse que les liquidites 0/20, perte au-dela plus basse 7/20
  treso_inter_60   avant_2022 : volatilite plus basse que les liquidites 20/20, perte au-dela plus basse 20/20
  treso_inter_60  depuis_2022 : volatilite plus basse que les liquidites 0/20, perte au-dela plus basse 7/20
  treso_long_80   avant_2022 : volatilite plus basse que les liquidites 20/20, perte au-dela plus basse 20/20
  treso_long_80  depuis_2022 : volatilite plus basse que les liquidites 0/20, perte au-dela plus basse 4/20
  treso_long_60   avant_2022 : volatilite plus basse que les liquidites 18/20, perte au-dela plus basse 20/20
  treso_long_60  depuis_2022 : volatilite plus basse que les liquidites 0/20, perte au-dela plus basse 2/20

Hypothese 2 : actions hors theme a 60 % cont

## 10. Significativité des écarts face à la série non couverte

Bootstrap circulaire par blocs d'un trimestre, 2 000 tirages, les deux séries tirées aux mêmes dates, comme à l'étape 3. Le Sharpe est calculé ici sur l'écart-type des rendements excédentaires.

In [13]:
BLOC, TIRAGES = 63, 2000

def excedents(v):
    r = v.pct_change().dropna()
    return (r - sans_risque.reindex(r.index)).to_numpy()

def sharpe_annuel(x):
    return x.mean() / x.std(ddof=1) * np.sqrt(252)

def vol_annuelle(x):
    return x.std(ddof=1) * np.sqrt(252)

lignes = []
for nom, strategies in resultats.items():
    b = excedents(strategies["non_couvert"][0])
    for strat, (v, _, _) in strategies.items():
        if strat == "non_couvert":
            continue
        a = excedents(v)
        for mesure, stat in [("volatilite", vol_annuelle), ("sharpe", sharpe_annuel)]:
            lignes.append({"serie": nom, "strategie": strat, "mesure": mesure,
                           **risque.ecart_bootstrap(a, b, stat, BLOC, TIRAGES)})
bootstrap = pd.DataFrame(lignes)
bootstrap["significatif_5pct"] = bootstrap.p_bilaterale < 0.05
bootstrap.to_csv(SORTIE / "couverture_bootstrap.csv", index=False, encoding="utf-8")
print(bootstrap[bootstrap.serie.isin(["P1_reeq", "P1_cons"])].round(4).to_string(index=False))
print()
vingt = bootstrap[bootstrap.serie.str.startswith("P")]
print(vingt.assign(hausse=vingt.ecart > 0).groupby(["mesure", "strategie"]).agg(
    significatifs=("significatif_5pct", "sum"),
    significatifs_hausse=("significatif_5pct", lambda s: int((s & vingt.loc[s.index, "ecart"].gt(0)).sum())),
    significatifs_baisse=("significatif_5pct", lambda s: int((s & vingt.loc[s.index, "ecart"].lt(0)).sum()))).to_string())

  serie      strategie     mesure   ecart     bas    haut  p_bilaterale  significatif_5pct
P1_reeq  liquidites_80 volatilite -0.0454 -0.0540 -0.0380         0.000               True
P1_reeq  liquidites_80     sharpe  0.0124 -0.0032  0.0271         0.133              False
P1_reeq  liquidites_60 volatilite -0.0891 -0.1041 -0.0760         0.000               True
P1_reeq  liquidites_60     sharpe  0.0233 -0.0069  0.0510         0.145              False
P1_reeq treso_inter_80 volatilite -0.0485 -0.0582 -0.0402         0.000               True
P1_reeq treso_inter_80     sharpe  0.0501  0.0186  0.0779         0.002               True
P1_reeq treso_inter_60 volatilite -0.0936 -0.1104 -0.0792         0.000               True
P1_reeq treso_inter_60     sharpe  0.1171  0.0414  0.1859         0.005               True
P1_reeq  treso_long_80 volatilite -0.0518 -0.0626 -0.0421         0.000               True
P1_reeq  treso_long_80     sharpe  0.0748  0.0133  0.1323         0.025               True

## 11. Sensibilité des puts à la pente de volatilité et à la fourchette

La phase 0 a fixé la pente à trois points et la fourchette à 5 % de la prime, sans les avoir mesurées. Les variantes encadrent ces deux choix, sur toutes les séries.

*Complément du 28 septembre 2026* : la variante à trois points sous le `VIX` est ajoutée après le contrôle contre l'indice PutWrite, parce qu'elle est la plus proche du niveau de prix observé à la monnaie. Elle ne remplace pas la base de la phase 0 dans les tableaux, pour que la trace de la décision initiale reste lisible.

In [14]:
lignes = []
for nom, niveau in series.items():
    r = niveau.pct_change()
    for niveau_k in (0.90, 0.95):
        for pente, fourchette in [(-0.03, FOURCHETTE), (0.0, FOURCHETTE), (PENTE, FOURCHETTE), (0.06, FOURCHETTE), (PENTE, 0.10)]:
            v, primes, encaisse = puts(r, betas[nom], niveau_k, pente, fourchette)
            lignes.append({"serie": nom, "strategie": f"put_{round(100 * niveau_k)}", "pente": pente,
                           "fourchette": fourchette, **mesures_courtes(v),
                           "cout_explicite": (primes - encaisse) / v.mean() / annees})
sensibilite = pd.DataFrame(lignes)
sensibilite.to_csv(SORTIE / "couverture_sensibilite_puts.csv", index=False, encoding="utf-8")
print(sensibilite[sensibilite.serie.isin(["P1_reeq", "P1_cons"])].round(4).to_string(index=False))
print()
print("medianes sur les vingt series :")
print(sensibilite[sensibilite.serie.str.startswith("P")].groupby(["strategie", "pente", "fourchette"])
      [["annualise", "vol_quotidienne", "repli_max", "es99", "cout_explicite"]].median().round(4).to_string())

  serie strategie  pente  fourchette  annualise  vol_quotidienne  repli_max   es99  sharpe  cout_explicite
P1_reeq    put_90  -0.03        0.05     0.1766           0.1869    -0.4967 0.0362  0.8634          0.0062
P1_reeq    put_90   0.00        0.05     0.1687           0.1848    -0.5049 0.0358  0.8346          0.0114
P1_reeq    put_90   0.03        0.05     0.1575           0.1826    -0.5161 0.0353  0.7898          0.0198
P1_reeq    put_90   0.06        0.05     0.1425           0.1802    -0.5286 0.0349  0.7252          0.0320
P1_reeq    put_90   0.03        0.10     0.1559           0.1826    -0.5194 0.0353  0.7819          0.0209
P1_reeq    put_95  -0.03        0.05     0.1662           0.1623    -0.4093 0.0314  0.9128          0.0208
P1_reeq    put_95   0.00        0.05     0.1379           0.1589    -0.4324 0.0309  0.7745          0.0441
P1_reeq    put_95   0.03        0.05     0.1050           0.1558    -0.4592 0.0305  0.5985          0.0725
P1_reeq    put_95   0.06        0.05 

## 12. Chiffres de synthèse repris dans le rapport

Les décomptes sur les vingt séries du bootstrap et la double lecture de `P4` exigée par la décision du 8 septembre 2026.

In [15]:
vingt = bootstrap[bootstrap.serie.str.startswith("P")]
decompte = vingt.groupby(["mesure", "strategie"]).apply(lambda g: pd.Series({
    "baisse_significative": int((g.significatif_5pct & (g.ecart < 0)).sum()),
    "hausse_significative": int((g.significatif_5pct & (g.ecart > 0)).sum())}))
print(decompte.to_string())
print()
p4 = pd.concat({"complete": mesures_df[mesures_df.serie.str.startswith("P4")].set_index(["serie", "strategie"]),
                "depuis_2004-08-19": sous_periodes[sous_periodes.periode == "depuis_2004-08-19"]
                .set_index(["serie", "strategie"])})
print(p4[["reduction_vol_quotidienne", "reduction_es99"]].unstack(0).round(3).to_string())
print()
print("repli maximal aggrave par le put a 90 %% : %d series sur 20"
      % int((mesures_df[mesures_df.serie.str.startswith("P") & (mesures_df.strategie == "put_90")].reduction_repli_max < 0).sum()))
print("SPY, 2000-2002 : %.1f %%" % (100 * (comparaisons.SPY.loc["2002-10-09"] / comparaisons.SPY.loc["2000-03-24"] - 1)))

                           baisse_significative  hausse_significative
mesure     strategie                                                 
sharpe     futur_100                          2                     0
           futur_50                           0                     4
           hors_theme_60                      0                     9
           hors_theme_80                      0                    10
           liquidites_60                      0                     2
           liquidites_80                      0                     2
           put_90                             6                     0
           put_95                            20                     0
           treso_inter_60                     0                    13
           treso_inter_80                     0                    14
           treso_long_60                      0                     2
           treso_long_80                      0                     8
volatilite futur_100

## 13. Sorties et manifeste

Les valeurs de toutes les stratégies sont conservées compressées, pour que l'étape 5 et les figures partent des mêmes séries.

In [16]:
chemin = SORTIE / "couverture_valeurs.csv.gz"
chemin.write_bytes(gzip.compress(table.to_csv(float_format="%.10g", lineterminator="\n").encode("utf-8"), mtime=0))
sorties = ["couverture_valeurs.csv.gz", "couverture_mesures.csv", "couverture_synthese.csv",
           "couverture_face_liquidites.csv", "couverture_correlations.csv", "couverture_sous_periodes.csv",
           "couverture_episodes.csv", "couverture_bootstrap.csv", "couverture_sensibilite_puts.csv"]
manifeste = {"produit_par": "src/couvrir_diversifier.ipynb",
             "decisions": "research/plan_projet.md, section VII",
             "strategies": STRATEGIES, "series": list(series), "pente_put": PENTE, "fourchette_put": FOURCHETTE, "pente_calibree_put_write": PENTE_CALIBREE,
             "cout_melange": cv.COUT_MELANGE, "cout_futur": cv.COUT_FUTUR,
             "instruments_sha256": {e["fichier"]: e["sha256"] for e in manifeste_instruments["instruments"]},
             "sorties_sha256": {s: empreinte(SORTIE / s) for s in sorties}}
(SORTIE / "couverture_manifest.json").write_text(json.dumps(manifeste, ensure_ascii=False, indent=2) + "\n",
                                                  encoding="utf-8")
print(f"{chemin.name} : {chemin.stat().st_size / 1e6:.1f} Mo")

couverture_valeurs.csv.gz : 10.1 Mo
